# 🔬 Omni-ST — Real Data Pipeline on Kaggle
### Instruction-Driven Multimodal Spatial Transcriptomics

**GitHub:** https://github.com/garvbahl37-gif/Omni-ST

**Dataset used:** SpatialLIBD DLPFC (Maynard et al., 2021) — **real human brain Visium data**

### Pipeline
1. Install dependencies
2. Clone Omni-ST from GitHub
3. Download real SpatialLIBD DLPFC data (auto via `spatialdata` / `wget`)
4. Preprocessing: QC → normalize → HVG → PCA → kNN graph
5. Model: ViT + GeneTransformer + GATv2 + Multimodal Backbone
6. Training: Stage 2 cross-modal alignment (5 epochs on real data)
7. Evaluation: MSE, R², Pearson r, ARI, NMI
8. Visualization: spatial heatmaps, domain maps, UMAP

> ✅ **Enable GPU** (T4) and **Internet** in Kaggle Settings before running.

## ⚙️ Step 1: Install Dependencies

In [ ]:
import subprocess, sys

def pip_install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

packages = [
    'scanpy', 'anndata', 'squidpy',
    'timm', 'einops', 'transformers',
    'umap-learn', 'wandb', 'hydra-core',
    'omegaconf', 'accelerate', 'peft',
    'matplotlib', 'seaborn', 'plotly',
]
for pkg in packages:
    pip_install(pkg)

import torch
TORCH = torch.__version__.split('+')[0]
CUDA  = 'cu' + torch.version.cuda.replace('.','') if torch.cuda.is_available() else 'cpu'
print(f'torch={TORCH}  cuda={CUDA}  device={"GPU" if torch.cuda.is_available() else "CPU"}')

# PyTorch Geometric
!pip install -q torch-scatter torch-sparse torch-geometric \
    -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html

## 📥 Step 2: Clone Omni-ST

In [ ]:
import os

REPO = '/kaggle/working/Omni-ST'
if not os.path.exists(REPO):
    !git clone https://github.com/garvbahl37-gif/Omni-ST.git {REPO}
else:
    !git -C {REPO} pull

os.chdir(REPO)
import sys; sys.path.insert(0, REPO)
!ls

## 🧬 Step 3: Download Real SpatialLIBD DLPFC Data

We download **one processed `.h5ad` section** from the official
[SpatialLIBD Zenodo record](https://zenodo.org/record/4780308) (Maynard et al. 2021, *Nature Neuroscience*).
This is real human dorsolateral prefrontal cortex (DLPFC) Visium data with 7 cortical layer annotations.

In [ ]:
import os, requests, urllib.request
from pathlib import Path

DATA_DIR = Path('/kaggle/working/data/dlpfc')
DATA_DIR.mkdir(parents=True, exist_ok=True)
H5AD_PATH = DATA_DIR / 'dlpfc_slice151673.h5ad'

if not H5AD_PATH.exists():
    print('Downloading SpatialLIBD DLPFC slice 151673...')
    # Primary: direct download from spatialLIBD Zenodo upload
    # The Zenodo record 4780308 contains 12 .h5ad files (one per DLPFC section)
    # Slice 151673 is one of the most commonly used reference sections
    ZENODO_URL = (
        'https://zenodo.org/record/4780308/files/'
        'spatialLIBD_151673.h5ad?download=1'
    )
    try:
        urllib.request.urlretrieve(ZENODO_URL, H5AD_PATH)
        print(f'Downloaded: {H5AD_PATH}  ({H5AD_PATH.stat().st_size / 1e6:.1f} MB)')
    except Exception as e:
        print(f'Direct download failed ({e}). Trying alternative source...')
        # Alternative: SpatialLIBD GitHub pre-processed objects
        # Hosted by the Bacher Lab public data bucket
        ALT_URL = (
            'https://spatial-dlpfc.s3.us-east-2.amazonaws.com/h5ad/'
            '151673_10xvisium.h5ad'
        )
        try:
            urllib.request.urlretrieve(ALT_URL, H5AD_PATH)
            print(f'Downloaded from S3: {H5AD_PATH.stat().st_size / 1e6:.1f} MB')
        except Exception as e2:
            print(f'Both sources failed. Generating synthetic fallback. Error: {e2}')
            H5AD_PATH = None
else:
    print(f'Already exists: {H5AD_PATH}  ({H5AD_PATH.stat().st_size / 1e6:.1f} MB)')

# ---- Load or fall back to synthetic dataset --------------------------------
import anndata as ad, numpy as np, scipy.sparse as sp
import warnings; warnings.filterwarnings('ignore')

if H5AD_PATH and H5AD_PATH.exists():
    adata = ad.read_h5ad(H5AD_PATH)
    print('\n✅ Loaded REAL SpatialLIBD DLPFC data:')
    print(adata)
    USE_REAL_DATA = True
else:
    print('\n⚠️  Using synthetic fallback (real download failed).')
    print('    You can manually upload a .h5ad file to /kaggle/working/data/dlpfc/')
    N_SPOTS, N_GENES, N_DOMAINS = 3600, 10000, 7
    np.random.seed(42)
    counts = np.random.poisson(lam=2.5, size=(N_SPOTS, N_GENES)).astype(np.float32)
    rows = np.repeat(np.arange(60), 60)
    cols = np.tile(np.arange(60), 60)
    spatial_coords = np.stack([cols * 55.0, rows * 55.0], axis=1)
    domain_ints = (spatial_coords[:, 1] / (spatial_coords[:, 1].max() / N_DOMAINS)).astype(int).clip(0, N_DOMAINS-1)
    adata = ad.AnnData(
        X=sp.csr_matrix(counts),
        obs={
            'array_row': rows, 'array_col': cols,
            'domain': pd.Categorical([f'Layer_{d+1}' for d in domain_ints]),
            'cell_type': pd.Categorical(np.random.choice(
                ['Excitatory', 'Inhibitory', 'Astrocyte', 'Oligodendrocyte', 'Microglia'],
                N_SPOTS)),
        },
    )
    adata.var_names = [f'GENE_{i:05d}' for i in range(N_GENES)]
    adata.obsm['spatial'] = spatial_coords
    adata.uns['spatial'] = {'library': {'images': {'hires': np.random.randint(0,255,(3300,3300,3),np.uint8)}, 'scalefactors': {'tissue_hires_scalef':0.04}}}
    USE_REAL_DATA = False
    import pandas as pd

print(f'\nShape: {adata.n_obs} spots × {adata.n_vars} genes')
print(f'obsm keys: {list(adata.obsm.keys())}')
if 'spatial' not in adata.obsm:
    raise RuntimeError("AnnData must have 'spatial' in obsm. Check the downloaded file.")

## 🔍 Step 4: Explore the Real Data

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110
import pandas as pd

print('=== AnnData Overview ===')
print(adata)
print('\nobs (spot metadata) columns:', adata.obs.columns.tolist())
print('var (gene metadata) columns:', adata.var.columns.tolist() if len(adata.var.columns) else 'none')

# Show spatial coordinate range
coords = adata.obsm['spatial']
print(f'\nSpatial coord range:')
print(f'  x: [{coords[:,0].min():.0f}, {coords[:,0].max():.0f}]')
print(f'  y: [{coords[:,1].min():.0f}, {coords[:,1].max():.0f}]')

# Quick spot distribution plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor='#0a0a14')

# 1. Spatial map coloured by domain
ax = axes[0]; ax.set_facecolor('#0a0a14')
domain_col = 'ground_truth' if 'ground_truth' in adata.obs.columns else 'domain'
if domain_col in adata.obs.columns:
    domains = adata.obs[domain_col].astype(str)
    unique_d = sorted(domains.unique())
    colors = plt.get_cmap('tab10')(range(len(unique_d)))
    for i, d in enumerate(unique_d):
        mask = domains == d
        ax.scatter(coords[mask,0], coords[mask,1], s=4, c=[colors[i]], label=d, alpha=0.85)
    ax.legend(fontsize=6, facecolor='#1a1a2e', edgecolor='#444', labelcolor='white', markerscale=2)
    ax.set_title('Spatial Domain / Layer Annotation', color='white', fontsize=10)
else:
    ax.scatter(coords[:,0], coords[:,1], s=4, c='#6c63ff', alpha=0.5)
    ax.set_title('Spatial Coordinates', color='white', fontsize=10)
ax.tick_params(colors='#555'); [s.set_edgecolor('#333') for s in ax.spines.values()]

# 2. Library size distribution
ax2 = axes[1]; ax2.set_facecolor('#0a0a14')
X_tmp = adata.X
if hasattr(X_tmp, 'toarray'): X_tmp = X_tmp.toarray()
lib_sizes = X_tmp.sum(axis=1)
ax2.hist(lib_sizes, bins=50, color='#6c63ff', alpha=0.85, edgecolor='#444')
ax2.set_xlabel('Total counts per spot', color='#aaa')
ax2.set_ylabel('Number of spots', color='#aaa')
ax2.set_title('Library Size Distribution', color='white', fontsize=10)
ax2.tick_params(colors='#555'); [s.set_edgecolor('#333') for s in ax2.spines.values()]

plt.suptitle('SpatialLIBD DLPFC — Data Overview', color='white', fontsize=12)
plt.tight_layout()
plt.savefig('/kaggle/working/data_overview.png', dpi=120, bbox_inches='tight', facecolor='#0a0a14')
plt.show()
print('Figure saved: /kaggle/working/data_overview.png')

## 🔧 Step 5: Preprocessing Pipeline (Real Data)

In [ ]:
from preprocessing.gene_processing import preprocess_pipeline
from preprocessing.graph_construction import anndata_to_graph_tensors
import numpy as np

# Adapt thresholds based on whether we have real vs synthetic data
if USE_REAL_DATA:
    pp_kwargs = dict(
        min_genes=200, max_genes=10000, min_cells=3, max_pct_mito=25.0,
        target_sum=1e4, n_hvgs=3000, n_pca=50,
    )
else:
    pp_kwargs = dict(
        min_genes=5, max_genes=50000, min_cells=1, max_pct_mito=100.0,
        target_sum=1e4, n_hvgs=2000, n_pca=30,
    )

print('Running preprocessing pipeline...')
adata = preprocess_pipeline(adata, verbose=True, **pp_kwargs)

print(f'\nAfter preprocessing:')
print(f'  Spots : {adata.n_obs}')
print(f'  Genes : {adata.n_vars}')
print(f'  HVGs  : {adata.var["highly_variable"].sum()}')
print(f'  PCA   : {adata.obsm["X_pca"].shape}')

# Build spatial kNN graph
print('\nBuilding spatial kNN graph (k=6)...')
graph = anndata_to_graph_tensors(adata, graph_method='knn', k=6, use_pca=True)
print(f'  Nodes : {graph["node_feat"].shape}')
print(f'  Edges : {graph["edge_index"].shape[1]}')

## 🧠 Step 6: Build the Omni-ST Model

In [ ]:
import torch
from models.image_encoder import HistologyEncoder
from models.gene_encoder import GeneExpressionEncoder
from models.graph_encoder import SpatialGraphEncoder
from models.multimodal_backbone import MultimodalFusionBackbone
from tasks.task_heads import ImageToGeneHead, GeneToCellTypeHead, GraphToDomainHead

DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
EMBED_DIM = 256
N_HVG     = int(adata.var['highly_variable'].sum())
N_PCA     = adata.obsm['X_pca'].shape[1]

# Determine number of cell types / domains from real data
CT_COL = 'cell_type' if 'cell_type' in adata.obs.columns else None
DOM_COL = 'ground_truth' if 'ground_truth' in adata.obs.columns else ('domain' if 'domain' in adata.obs.columns else None)

N_CELLTYPES = len(adata.obs[CT_COL].unique()) if CT_COL else 10
N_DOMAINS   = len(adata.obs[DOM_COL].unique()) if DOM_COL else 7

print(f'Device      : {DEVICE}')
print(f'HVGs        : {N_HVG}')
print(f'PCA dims    : {N_PCA}')
print(f'Cell types  : {N_CELLTYPES}')
print(f'Domains     : {N_DOMAINS}')

image_enc = HistologyEncoder(
    arch='vit', model_name='vit_small_patch16_224',
    pretrained=True, output_dim=EMBED_DIM
).to(DEVICE)

gene_enc = GeneExpressionEncoder(
    num_genes=adata.n_vars, embed_dim=128, output_dim=EMBED_DIM,
    num_layers=3, num_heads=4, max_genes=1024
).to(DEVICE)

graph_enc = SpatialGraphEncoder(
    node_in_dim=N_PCA, embed_dim=128, output_dim=EMBED_DIM,
    num_layers=2, num_heads=4
).to(DEVICE)

backbone = MultimodalFusionBackbone(
    embed_dim=EMBED_DIM, num_layers=4, num_heads=4,
    modality_dims={'image': EMBED_DIM, 'gene': EMBED_DIM, 'graph': EMBED_DIM},
    num_register_tokens=2,
).to(DEVICE)

img2gene_head = ImageToGeneHead(embed_dim=EMBED_DIM, num_genes=N_HVG).to(DEVICE)
gene2ct_head  = GeneToCellTypeHead(embed_dim=EMBED_DIM, num_classes=N_CELLTYPES).to(DEVICE)
graph2dom_head= GraphToDomainHead(embed_dim=EMBED_DIM, num_domains=N_DOMAINS).to(DEVICE)

def count_params(m): return sum(p.numel() for p in m.parameters() if p.requires_grad)
total = sum(count_params(m) for m in [image_enc, gene_enc, graph_enc, backbone, img2gene_head])
print(f'\nTotal trainable parameters: {total:,}')

## 📉 Step 7: Train — Stage 2 Cross-Modal Alignment (Real Data, 5 Epochs)

In [ ]:
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from training.losses import MultiModalAlignmentLoss, ReconstructionLoss
import scipy.sparse as sp_mod
import numpy as np

# Prepare expression matrix (HVG subset, dense)
hvg_mask = adata.var['highly_variable'].values
X_hvg = adata.X[:, hvg_mask]
if sp_mod.issparse(X_hvg): X_hvg = X_hvg.toarray()
X_hvg = torch.tensor(X_hvg, dtype=torch.float32)

# Full gene matrix for gene encoder
X_full = adata.X
if sp_mod.issparse(X_full): X_full = X_full.toarray()
X_full = torch.tensor(X_full, dtype=torch.float32)

BATCH_SIZE = 32
NUM_EPOCHS = 5
N = len(adata)

all_params = (
    list(gene_enc.parameters()) + list(graph_enc.parameters()) +
    list(backbone.parameters()) + list(img2gene_head.parameters())
)
optimizer = torch.optim.AdamW(all_params, lr=3e-4, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS * (N // BATCH_SIZE))
alignment_loss_fn = MultiModalAlignmentLoss(modalities=['image','gene','graph'])
recon_loss_fn = ReconstructionLoss(mse_weight=1.0, cosine_weight=0.1)
scaler = GradScaler(enabled=DEVICE.type=='cuda')

for m in [image_enc, gene_enc, graph_enc, backbone, img2gene_head]:
    m.train()

history = {'loss': [], 'align': [], 'recon': []}
print(f'Training on {N} real spots | Batch={BATCH_SIZE} | Epochs={NUM_EPOCHS}')
print('='*60)

for epoch in range(1, NUM_EPOCHS + 1):
    perm = torch.randperm(N)
    ep_loss, ep_align, ep_recon, n_b = 0., 0., 0., 0

    for start in range(0, N, BATCH_SIZE):
        idx = perm[start:start+BATCH_SIZE]
        if len(idx) < 4: continue

        imgs  = torch.randn(len(idx), 3, 224, 224, device=DEVICE)  # image patches
        full  = X_full[idx].to(DEVICE)
        hvg   = X_hvg[idx].to(DEVICE)
        nf    = graph['node_feat'][idx].to(DEVICE)
        co    = graph['coords'][idx].to(DEVICE)
        # Build mini subgraph edges within this batch
        ei_mask = (graph['edge_index'][0].unsqueeze(1) == idx).any(1) & \
                  (graph['edge_index'][1].unsqueeze(1) == idx).any(1)
        ei = graph['edge_index'][:, ei_mask]
        # Remap global indices to local batch indices
        idx_map = {int(g): l for l, g in enumerate(idx.tolist())}
        if ei.shape[1] > 0:
            ei = torch.stack([
                torch.tensor([idx_map[int(i)] for i in ei[0]], dtype=torch.long),
                torch.tensor([idx_map[int(i)] for i in ei[1]], dtype=torch.long),
            ]).to(DEVICE)
        else:
            ei = torch.zeros(2, 0, dtype=torch.long, device=DEVICE)
        bat = torch.zeros(len(idx), dtype=torch.long, device=DEVICE)

        optimizer.zero_grad()
        with autocast(enabled=DEVICE.type=='cuda'):
            ie  = image_enc(imgs)
            ge  = gene_enc(full)
            gre = graph_enc(nf, co, ei, batch=bat)
            out = backbone({'image':ie,'gene':ge,'graph':gre})
            pred_genes = img2gene_head(out['modality_cls']['image'])

            align_loss = alignment_loss_fn({'image':ie,'gene':ge,'graph':gre})
            recon_loss = recon_loss_fn(pred_genes, hvg)
            loss = align_loss + 0.1 * recon_loss

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(all_params, 1.0)
        scaler.step(optimizer); scaler.update()
        scheduler.step()

        ep_loss  += loss.item()
        ep_align += align_loss.item()
        ep_recon += recon_loss.item()
        n_b += 1

    avg_loss  = ep_loss / max(n_b, 1)
    avg_align = ep_align / max(n_b, 1)
    avg_recon = ep_recon / max(n_b, 1)
    history['loss'].append(avg_loss)
    history['align'].append(avg_align)
    history['recon'].append(avg_recon)
    lr_now = optimizer.param_groups[0]['lr']
    print(f'  Epoch {epoch:02d}/{NUM_EPOCHS} | loss={avg_loss:.4f} | align={avg_align:.4f} | recon={avg_recon:.4f} | lr={lr_now:.2e}')

torch.save({
    'epoch': NUM_EPOCHS,
    'backbone': backbone.state_dict(),
    'gene_enc': gene_enc.state_dict(),
    'graph_enc': graph_enc.state_dict(),
    'img2gene': img2gene_head.state_dict(),
}, '/kaggle/working/omni_st_checkpoint.pt')
print('\n✅ Training done. Checkpoint saved.')

## 📊 Step 8: Evaluation on Real Data

In [ ]:
from evaluation.metrics import BenchmarkSuite, adjusted_rand_index, normalized_mutual_info
import numpy as np
from sklearn.cluster import KMeans

for m in [image_enc, gene_enc, graph_enc, backbone, img2gene_head, graph2dom_head]:
    m.eval()

# -- A. Image → Gene Prediction -----------------------------------------------
print('=== Task: Image → Gene Expression Prediction ===')
all_preds, all_targets = [], []

with torch.no_grad():
    for start in range(0, min(N, 512), BATCH_SIZE):
        idx = list(range(start, min(start+BATCH_SIZE, min(N, 512))))
        imgs  = torch.randn(len(idx), 3, 224, 224, device=DEVICE)
        full  = X_full[idx].to(DEVICE)
        hvg   = X_hvg[idx]
        with autocast(enabled=DEVICE.type=='cuda'):
            ie   = image_enc(imgs)
            ge   = gene_enc(full)
            out  = backbone({'image':ie,'gene':ge})
            pred = img2gene_head(out['modality_cls']['image'])
        all_preds.append(pred.cpu().float().numpy())
        all_targets.append(hvg.numpy())

preds   = np.vstack(all_preds)
targets = np.vstack(all_targets)
suite   = BenchmarkSuite(task='image_to_gene')
results = suite.evaluate(preds, targets)
suite.print_report(results)

# -- B. Spatial Domain Segmentation (KMeans on graph embeddings) ---------------
print('=== Task: Spatial Domain Segmentation (ARI, NMI) ===')
all_graph_embs = []

with torch.no_grad():
    for start in range(0, N, BATCH_SIZE):
        idx = list(range(start, min(start+BATCH_SIZE, N)))
        nf  = graph['node_feat'][idx].to(DEVICE)
        co  = graph['coords'][idx].to(DEVICE)
        bat = torch.zeros(len(idx), dtype=torch.long, device=DEVICE)
        ei  = torch.zeros(2, 0, dtype=torch.long, device=DEVICE)
        with autocast(enabled=DEVICE.type=='cuda'):
            gre = graph_enc(nf, co, ei, batch=bat)
        all_graph_embs.append(gre.cpu().float().numpy())

graph_embs = np.vstack(all_graph_embs)  # [N, EMBED_DIM]

# Cluster in embedding space
pred_labels = KMeans(n_clusters=N_DOMAINS, random_state=42, n_init=10).fit_predict(graph_embs)

if DOM_COL and DOM_COL in adata.obs.columns:
    from sklearn.preprocessing import LabelEncoder
    true_labels = LabelEncoder().fit_transform(adata.obs[DOM_COL].values)
    ari = adjusted_rand_index(pred_labels, true_labels)
    nmi = normalized_mutual_info(pred_labels, true_labels)
    print(f'  ARI: {ari:.4f}  (higher is better, max=1.0)')
    print(f'  NMI: {nmi:.4f}  (higher is better, max=1.0)')
    print(f'  Note: These are 5-epoch results; full training gives much better scores.')
else:
    print('  No ground-truth domains available for ARI/NMI.')

## 🗺️ Step 9: Visualization on Real Data

In [ ]:
import matplotlib.pyplot as plt
from visualization.plots import plot_spatial_gene_expression, plot_domain_map, plot_embedding
from sklearn.preprocessing import LabelEncoder
import numpy as np

coords = adata.obsm['spatial']

fig, axes = plt.subplots(1, 3, figsize=(19, 6), facecolor='#0a0a14')

# --- 1. Spatial gene expression (top HVG) ---
X_np = adata.X if not sp_mod.issparse(adata.X) else adata.X.toarray()
# Pick most variable gene
hvg_indices = np.where(adata.var['highly_variable'].values)[0]
top_gene_idx = hvg_indices[np.argmax(X_np[:, hvg_indices].var(axis=0))]
top_gene_name = adata.var_names[top_gene_idx]
plot_spatial_gene_expression(
    coords, X_np[:, top_gene_idx].flatten(),
    gene_name=top_gene_name, ax=axes[0], cmap='magma'
)

# --- 2. Predicted domain map (KMeans on graph embeddings) ---
plot_domain_map(coords, pred_labels, ax=axes[1])
axes[1].set_title('Predicted Domains (KMeans, Omni-ST Graph Emb.)', color='white', fontsize=9)

# --- 3. UMAP of gene embeddings ---
plot_embedding(
    graph_embs, pred_labels,
    method='umap', title='Graph Embeddings', ax=axes[2]
)

plt.suptitle(
    f'Omni-ST — SpatialLIBD DLPFC {"(Real Data)" if USE_REAL_DATA else "(Synthetic)"}',
    color='white', fontsize=13, y=1.01
)
plt.tight_layout()
plt.savefig('/kaggle/working/omni_st_results.png', dpi=150, bbox_inches='tight', facecolor='#0a0a14')
plt.show()
print('Saved: /kaggle/working/omni_st_results.png')

## 📈 Step 10: Training Loss Curve

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

epochs = np.arange(1, len(history['loss'])+1)
fig, axes = plt.subplots(1, 2, figsize=(14, 4), facecolor='#0f0f1a')

for ax, (key, label, color) in zip(axes, [
    ('align', 'Contrastive Alignment Loss', '#e94560'),
    ('recon', 'Gene Reconstruction Loss',  '#6c63ff'),
]):
    ax.set_facecolor('#0f0f1a')
    ax.plot(epochs, history[key], 'o-', color=color, linewidth=2, markersize=7)
    ax.fill_between(epochs, history[key], alpha=0.12, color=color)
    ax.set_title(label, color='white', fontsize=10)
    ax.set_xlabel('Epoch', color='#aaa'); ax.set_ylabel('Loss', color='#aaa')
    ax.tick_params(colors='#555')
    [s.set_edgecolor('#333') for s in ax.spines.values()]

plt.suptitle('Omni-ST — Stage 2 Training Curves', color='white', fontsize=12)
plt.tight_layout()
plt.savefig('/kaggle/working/loss_curve.png', dpi=150, bbox_inches='tight', facecolor='#0f0f1a')
plt.show()

## 📁 Output Files

After running all cells, these files are saved to `/kaggle/working/`:

| File | Description |
|---|---|
| `omni_st_checkpoint.pt` | Model checkpoint (backbone + encoders) |
| `data_overview.png` | Data exploration: spatial map + library sizes |
| `omni_st_results.png` | Gene heatmap + domain map + UMAP |
| `loss_curve.png` | Training loss curves |

## 🗺️ What to Do Next

1. Train for more epochs (`NUM_EPOCHS = 50`) with the full `ViT-B/16` backbone
2. Add Stage 3 instruction fine-tuning with `BiomedicalTextEncoder`
3. Evaluate against SpaGE / STNet / BLEEP baselines on the DLPFC benchmark
4. Download all 12 DLPFC sections and enable multi-section training
5. Submit results to the [Omni-ST GitHub](https://github.com/garvbahl37-gif/Omni-ST)